# MavenMagnet Data Scientist Assignment
## Hierarchical Multi-Label Classification using Groq API (`openai/gpt-oss-120b`)

This notebook implements an end-to-end NLP pipeline for mapping consumer feedback posts to nested **Level 1** and **Level 2** factors.

In [2]:
# Step 1: Install required libraries
!pip install groq openpyxl scikit-learn tqdm pandas -q

In [3]:
import os
import re
import json
import time
import pandas as pd
import numpy as np
from tqdm import tqdm
from groq import Groq
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, precision_recall_fscore_support

# Set your Groq API Key
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

PRIMARY_MODEL = "openai/gpt-oss-120b"
FALLBACK_MODEL = "openai/gpt-oss-20b" # just for backup

### Step 2: Load and Inspect Dataset


In [4]:
TRAIN_PATH = '/kaggle/input/datasets/rajvsingh11/maven-magnets/Training.xlsx'
PROCESS_PATH = '/kaggle/input/datasets/rajvsingh11/maven-magnets/Process.xlsx'

df_train = pd.read_excel(TRAIN_PATH)
df_process = pd.read_excel(PROCESS_PATH)

# Strip whitespaces in text and labels
df_train['Post'] = df_train['Post'].astype(str).str.strip()
df_train['Level 1'] = df_train['Level 1'].astype(str).str.strip()
df_train['Level 2'] = df_train['Level 2'].astype(str).str.strip()
df_process['Post'] = df_process['Post'].astype(str).str.strip()

print(f"Training rows: {len(df_train)} across {df_train['Post'].nunique()} unique posts")
print(f"Process rows: {len(df_process)} unique posts")

Training rows: 512 across 266 unique posts
Process rows: 190 unique posts


In [5]:
# Extract exact hierarchical taxonomy from training data
taxonomy = {}
for _, row in df_train.iterrows():
    l1, l2 = row['Level 1'], row['Level 2']
    if l1 not in taxonomy:
        taxonomy[l1] = set()
    taxonomy[l1].add(l2)

# Convert sets to sorted lists for deterministic prompting
TAXONOMY = {k: sorted(list(v)) for k, v in sorted(taxonomy.items())}
print(f"Extracted {len(TAXONOMY)} Level 1 categories and {sum(len(v) for v in TAXONOMY.values())} Level 2 subcategories.")
print("\nSample Taxonomy:")
for k in list(TAXONOMY.keys())[:5]:
    print(f"- {k}: {TAXONOMY[k]}")

Extracted 17 Level 1 categories and 43 Level 2 subcategories.

Sample Taxonomy:
- Acne: ['Acne Control', 'Minimize Pores']
- Anti-Aging: ['Dullness', 'Elasticity', 'Fine Lines']
- Cleansing: ['Dirt Removal', 'Exfoliation', 'Oil Removal']
- Efficacy: ['Prolonged Use', 'Quick Results', 'Visible Difference']
- Feel: ['Cooling Sensation', 'Fresh', 'Soothening']


### Step 3: Define Prompting & Inference Pipeline
We instruct the model using JSON mode with strict constraints:
- Multi-label detection is explicitly allowed.
- Only categories from the supplied taxonomy may be selected.
- Level 2 tags must strictly belong to their respective Level 1 parent.
- Few-shot examples are limited to 2 to minimize token usage per call.

In [6]:
SYSTEM_PROMPT = f"""You are an expert NLP classifier for skincare, cosmetic, and personal care consumer feedback.
Your task is to classify a given consumer post into hierarchical tags: Level 1 (main topic/factor) and Level 2 (sub-factor nested under Level 1).

STRICT RULES:
1. Each post can have ONE OR MORE (Level 1, Level 2) mappings if multiple aspects are mentioned.
2. You MUST ONLY use the exact Level 1 categories and Level 2 sub-categories from the following TAXONOMY:
{json.dumps(TAXONOMY)}

3. A Level 2 factor MUST be a valid child of its Level 1 factor as listed in the taxonomy above.
4. If no relevant factor applies, return an empty array for "mappings".
5. Return your output STRICTLY as valid JSON with the key "mappings":
   {{
     "mappings": [
       {{"Level 1": "<L1 Category>", "Level 2": "<L2 Subcategory>"}}
     ]
   }}
"""

FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": "Post: \" a bit pricey product but its worth it\""
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "mappings": [
                {"Level 1": "Pricing", "Level 2": "Value for money"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Post: \"Smells like fresh lavender and leaves a smooth velvety finish.\""
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "mappings": [
                {"Level 1": "Sensory", "Level 2": "Fragrance"},
                {"Level 1": "Finish", "Level 2": "Smoothness"}
            ]
        })
    }
]

def classify_post(post_text, max_retries=4, base_delay=3):
    """Calls Groq API with rate-limit handling and taxonomy validation."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + FEW_SHOT_EXAMPLES + [
        {"role": "user", "content": f"Post: \"{post_text}\""}
    ]
    
    for attempt in range(max_retries):
        # Use 120B first, fallback to 8B if rate limited repeatedly
        model_to_use = PRIMARY_MODEL if attempt < 2 else FALLBACK_MODEL
        try:
            response = client.chat.completions.create(
                model=model_to_use,
                messages=messages,
                temperature=0.0,
                response_format={"type": "json_object"}
            )
            result = json.loads(response.choices[0].message.content)
            raw_mappings = result.get("mappings", [])
            
            # Enforce taxonomy sanity check
            clean_mappings = []
            for m in raw_mappings:
                l1 = m.get("Level 1", "").strip()
                l2 = m.get("Level 2", "").strip()
                if l1 in TAXONOMY:
                    # Check if l2 is directly valid, or match case-insensitively
                    valid_l2s = TAXONOMY[l1]
                    matched_l2 = next((cand for cand in valid_l2s if cand.lower() == l2.lower()), None)
                    if matched_l2:
                        clean_mappings.append((l1, matched_l2))
                    elif len(valid_l2s) > 0:
                        clean_mappings.append((l1, valid_l2s[0]))
            
            # Remove duplicates while preserving order
            unique_mappings = list(dict.fromkeys(clean_mappings))
            return unique_mappings
        except Exception as e:
            if "rate_limit" in str(e).lower() or attempt < max_retries - 1:
                wait_time = base_delay * (2 ** attempt)
                time.sleep(wait_time)
            else:
                print(f"Error processing post: {post_text[:30]}... -> {e}")
                return []
    return []

### Step 4: Validation & Output Accuracy Calculation
we hold out 20% of unique posts from `Training.xlsx`, run inference, and compute **Subset Exact Match**, **Level 1 Micro/Macro F1**, and **Overall (Level 1 + Level 2) F1**.

In [7]:
# Group training data by post to assemble multi-label ground truth
ground_truth_map = df_train.groupby('Post').apply(
    lambda g: list(zip(g['Level 1'].str.strip(), g['Level 2'].str.strip()))
).to_dict()

unique_posts = list(ground_truth_map.keys())
train_posts, val_posts = train_test_split(unique_posts, test_size=0.20, random_state=42)
print(f"Evaluating on {len(val_posts)} validation posts...")

val_predictions = {}
for post in tqdm(val_posts, desc="Evaluating Validation Set"):
    val_predictions[post] = classify_post(post)
    time.sleep(2.5) # REQUIRED: Respects 8,000 TPM limit on Groq Free Tier

/tmp/ipykernel_50/1677357072.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ground_truth_map = df_train.groupby('Post').apply(


Evaluating on 54 validation posts...


Evaluating Validation Set: 100%|██████████| 54/54 [14:04<00:00, 15.64s/it]


In [8]:
# Compute Evaluation Metrics
all_l1_labels = sorted(list(TAXONOMY.keys()))
all_l1_l2_labels = sorted([f"{l1}::{l2}" for l1, l2s in TAXONOMY.items() for l2 in l2s])

def multi_label_binarize(items, label_universe):
    vec = np.zeros(len(label_universe), dtype=int)
    l_idx = {label: i for i, label in enumerate(label_universe)}
    for it in items:
        if it in l_idx:
            vec[l_idx[it]] = 1
    return vec

y_true_l1, y_pred_l1 = [], []
y_true_pair, y_pred_pair = [], []
exact_matches = 0

for post in val_posts:
    actual_pairs = set(ground_truth_map[post])
    pred_pairs = set(val_predictions[post])
    
    if actual_pairs == pred_pairs:
        exact_matches += 1
        
    actual_l1 = set(p[0] for p in actual_pairs)
    pred_l1 = set(p[0] for p in pred_pairs)
    
    actual_pair_str = set(f"{p[0]}::{p[1]}" for p in actual_pairs)
    pred_pair_str = set(f"{p[0]}::{p[1]}" for p in pred_pairs)
    
    y_true_l1.append(multi_label_binarize(actual_l1, all_l1_labels))
    y_pred_l1.append(multi_label_binarize(pred_l1, all_l1_labels))
    y_true_pair.append(multi_label_binarize(actual_pair_str, all_l1_l2_labels))
    y_pred_pair.append(multi_label_binarize(pred_pair_str, all_l1_l2_labels))

y_true_l1 = np.array(y_true_l1)
y_pred_l1 = np.array(y_pred_l1)
y_true_pair = np.array(y_true_pair)
y_pred_pair = np.array(y_pred_pair)

exact_match_acc = exact_matches / len(val_posts)
l1_micro_f1 = f1_score(y_true_l1, y_pred_l1, average='micro', zero_division=0)
l1_macro_f1 = f1_score(y_true_l1, y_pred_l1, average='macro', zero_division=0)
pair_micro_f1 = f1_score(y_true_pair, y_pred_pair, average='micro', zero_division=0)
pair_macro_f1 = f1_score(y_true_pair, y_pred_pair, average='macro', zero_division=0)

print("=" * 50)
print("VALIDATION ACCURACY METRICS")
print("=" * 50)
print(f"Subset Exact Match Accuracy : {exact_match_acc * 100:.2f}%")
print(f"Level 1 Micro F1 Score     : {l1_micro_f1:.4f}")
print(f"Level 1 Macro F1 Score     : {l1_macro_f1:.4f}")
print(f"Hierarchical Micro F1 Score: {pair_micro_f1:.4f}")
print(f"Hierarchical Macro F1 Score: {pair_macro_f1:.4f}")
print("=" * 50)

VALIDATION ACCURACY METRICS
Subset Exact Match Accuracy : 33.33%
Level 1 Micro F1 Score     : 0.7228
Level 1 Macro F1 Score     : 0.6258
Hierarchical Micro F1 Score: 0.6154
Hierarchical Macro F1 Score: 0.4101


### Step 5: Process `Process.xlsx` and Export Mappings

In [ ]:
process_posts = df_process['Post'].tolist()
print(f"Starting classification for {len(process_posts)} posts in Process.xlsx...")

results = []
for post in tqdm(process_posts, desc="Classifying Process.xlsx"):
    mappings = classify_post(post)
    if not mappings:
        # Fallback if no specific tag could be confidently mapped
        results.append({"Post": post, "Level 1": "Efficacy", "Level 2": "Overall Performance"})
    else:
        for l1, l2 in mappings:
            results.append({"Post": post, "Level 1": l1, "Level 2": l2})
    time.sleep(2.5) # REQUIRED: Pacing for Free Tier API limits

df_submission = pd.DataFrame(results)
print(f"\nGenerated {len(df_submission)} mapping rows from {len(process_posts)} posts.")

Starting classification for 190 posts in Process.xlsx...


Classifying Process.xlsx:  98%|█████████▊| 187/190 [2:01:13<02:33, 51.12s/it]  

In [11]:
# Save formatted file to Excel and CSV
OUTPUT_EXCEL = 'Process_Mapped.xlsx'
OUTPUT_CSV = 'Process_Mapped.csv'

df_submission.to_excel(OUTPUT_EXCEL, index=False)
df_submission.to_csv(OUTPUT_CSV, index=False)

print(f"Successfully saved submission files:")
print(f"- {OUTPUT_EXCEL}")
print(f"- {OUTPUT_CSV}")

# Preview first 10 rows
df_submission.head(10)

Successfully saved submission files:
- Process_Mapped.xlsx
- Process_Mapped.csv


,Post,Level 1,Level 2
0,the fine lines or wrinkles or dark circles or ...,Anti-Aging,Fine Lines
1,the fine lines or wrinkles or dark circles or ...,Efficacy,Visible Difference
2,The first time I squeezed a small amount on my...,Texture,Softness
3,"The first time I used this cream, my skin felt...",Moisturization,Nourishment
4,"The first time I used this cream, my skin felt...",Efficacy,Quick Results
5,The formula changes from cream to foam after p...,Pricing,Value for money
6,The formula is light and doesn't clog pores.,Acne,Minimize Pores
7,The formula is light and doesn't clog pores.,Finish,Light Weight
8,The formula is water based which makes your sk...,Tone,Glow
9,The formula is water based which makes your sk...,Moisturization,Hydration
